# Executor Memory Sizing and Overhead

**Module**: 06_memory_and_storage  
**Notebook**: 02_executor_memory.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Sizing spark.executor.memory and spark.executor.memoryOverhead, and Docker container memory limits.
- **Why does this topic exist?**: Incorrect sizing causes Linux kernel OOM kills (Exit Code 137).
- **What problem does it solve?**: Container killed because off-heap PySpark memory was ignored.
- **Where is it used in real-world Data Engineering?**: Production container resource requests in Kubernetes and Docker Compose.

---

## 2. Docker Container vs Spark Heap

`Container Limit >= spark.executor.memory + spark.executor.memoryOverhead + OS Buffer`

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Executor_Memory_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
overhead_factor = 0.10 # default 10%
exec_mem_mb = 2048 # 2 GB
calculated_overhead = max(384, int(exec_mem_mb * overhead_factor))
print(f"Configured Heap:     {exec_mem_mb} MB")
print(f"Calculated Overhead: {calculated_overhead} MB")
print(f"Total Process RAM:   {exec_mem_mb + calculated_overhead} MB")

---

## Experiment: Verifying Worker Capacity

- **1. What are we testing?**: Check worker capacity vs executor size in our cluster.
- **2. What do we expect?**: Worker has 4GB; Executor has 2GB -> 1-2 executors per worker.

In [ ]:
print('Worker memory from env: 4g, Executor memory: 2g')

**3. What happened?**: Matched Docker Compose configuration.  
**4. Why did it happen?**: Configured in .env.  
**5. Practical Engineering Lesson**: Ensure total executor requests do not exceed worker host capacity.

---

## Practical Exercises

### Exercise 1 (Beginner): Calculate Overhead for 8GB
**Problem Statement**: Compute overhead for 8GB heap.  
**Input / Context**: 8192 MB

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Exit Code 137 Meaning
**Problem Statement**: Explain exit code 137.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): PySpark Overhead
**Problem Statement**: Why PySpark needs higher overhead.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Calculate Overhead for 8GB):
```python
print(max(384, int(8192 * 0.10)))
```

#### Solution 2 (Exit Code 137 Meaning):
```python
print('Exit Code 137 = 128 + 9 (SIGKILL) = Container OOMKilled by Linux kernel')
```

#### Solution 3 (PySpark Overhead):
```python
print('PySpark spawns Python worker daemons outside the JVM heap.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Overhead**: Off-heap buffer for PySpark, Netty, Metaspace.
- **Docker Buffer**: Always provide a safety margin over heap.

### Essential Technical Terms
- `memoryOverhead`: Off-heap process allocation.
- `OOMKilled`: Kernel termination of process exceeding cgroup limit.

### Common Engineering Mistakes
- **Setting Docker limit equal to JVM heap**: Results in immediate container kill.

### Interview Questions on This Topic
- **Q: What causes Exit Code 137 in Spark?**
  - *A*: The Linux kernel OOM killer terminating the container because combined memory exceeded Docker memory limits.

### What to Study Next
Open 03_storage_and_execution_memory.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')